# 05. Forecasting, anomaly scores, and local explanations

We will train a separate past-to-future model, calibrate a normal-only score, and inspect which time-joint targets contribute to a prediction error.

## Run this notebook independently

From `experiments/aaai`, run:

```bash
uv sync
uv run jupyter lab
```

Select the kernel from this environment. Python 3.12 and all notebook libraries are declared in `pyproject.toml`. Every notebook repeats its setup and loads its own inputs. No variable from another notebook is required.

A fresh installation defaults to `AAAI_DEMO=1` for a small synthetic software check. This prepared workspace uses `AAAI_DEMO=0` after real-data ingestion and validation. Its condition parameters are invented and its scores are not research evidence. Set `AAAI_DEMO=0` in the repository root `.env` after the ingestion workflow has produced a complete pose corpus. All notebooks use the same root `.env`. No API key is needed for JEPA or Random Forest training. Restart the kernel after changing configuration.

Only the root environment file contains credentials. `load_env` and `find_env` below are aliases for the real `python-dotenv` functions, `load_dotenv` and `find_dotenv`.

## Historical tutorial and current study

This notebook preserves the earlier tutorial's code, observations and experimental scope. Its original result identities remain historical; they are not measurements of the new retained-predictor or U/J/A study. Start the current independent tutorial sequence at [Notebook 12](12_observation_population_and_video_context.ipynb), then continue through [Notebook 17](17_failure_analysis_and_manuscript_decision.ipynb). The [current protocol](docs/iteration02/protocol.md) and [current results](docs/iteration02/results.md) identify the verified new run. The [draft01 manuscript](docs/draft01/manuscript.md) remains the record of the completed earlier study.

The current primary question is normal versus the four requested condition families, with all other folders retained in a mandatory broad secondary analysis. It tests opposite-leg information beyond own-leg geometry and extra capacity. Uniform masking remains the reference, while J and A are declared motion-independent alternatives with matched budgets. Read the newer notebooks for their support rules and fixed evaluation banks. Do not combine old and new scores or denominators.

In [ ]:
from pathlib import Path
import os, sys
# Find the repository from either its root or this notebook directory.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'ambient').is_dir() and (p / 'data/gavd').is_dir())
BASE = ROOT / 'experiments/aaai'
os.chdir(ROOT)
from dotenv import load_dotenv as load_env, find_dotenv as find_env
load_env(find_env())  # requested spelling, using aliases for python-dotenv's real API
os.chdir(BASE)
if str(BASE) not in sys.path: sys.path.insert(0, str(BASE))
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from aaai_gait.settings import Settings
from aaai_gait.data import build_manifest, grouped_split, LABELS
from aaai_gait.media import show_video_context, plot_motion
settings = Settings()
torch.set_num_threads(2)
manifest = grouped_split(build_manifest(ROOT), seed=settings.seed)
print('Mode:', 'SYNTHETIC SOFTWARE DEMONSTRATION' if settings.demo else 'GAVD RESEARCH RUN')
print('Current annotation sequences:', len(manifest), 'unique sources:', manifest.source_id.nunique())

## Connect the experiment to walking

Watch a short annotated normal training sequence when local media is available. The preview is an observed video. Synthetic arrays used below are a separate software demonstration.

In [ ]:
video_row = show_video_context(manifest, frame_number_base=settings.frame_number_base)

In [ ]:
from aaai_gait.workflow import prepare_corpus
x, meta, provenance = prepare_corpus(settings)
print('Window tensor:', x.shape, '[windows, frames, selected joints, XY]')
display(meta.groupby(['split','label']).agg(windows=('sequence_id','size'), sequences=('sequence_id','nunique'), sources=('source_id','nunique')))

## 1. Separate completion from forecasting

Random masked completion can observe both earlier and later frames. A forecasting experiment hides every joint in the future half. The online encoder receives only the past half. Learned future time and joint positions are allowed because they do not contain future coordinates. The teacher sees all observed tokens in the window to construct training targets. Missing patches are excluded from its attention keys and the prediction loss.

The shared preprocessing estimates scale from the observed first half. Cohort admission also checks future observation coverage, so this is retrospective evaluation on evaluable futures rather than a streaming admission policy. This prevents future torso dimensions from entering past inputs. Prefix normalization assumes the configured forecast boundary remains at half the window. Changing that boundary requires updating preprocessing as well.

![Forecasting](images/forecast.svg)

In [ ]:
from aaai_gait.workflow import normal_anchor
from aaai_gait.model import forecast_mask
forecast_model,history=normal_anchor(x,meta,provenance,settings,mode='forecast')
mask=forecast_mask(1,forecast_model.cfg).reshape(8,16)
assert not mask[:4].any() and mask[4:].all()
print('Observed patches:',(~mask).any(1).nonzero().ravel().tolist())
display(pd.DataFrame(history))

## 2. Define predictive surprise

For each hidden token, we compare the target and predicted feature distributions using `KL(target || prediction)`. This subtracts the target entropy from cross-entropy, so a broad teacher distribution does not create a large score by itself. Masked completion averages a fixed bank of eight uniform masks. Every window uses the same bank, making scores independent of batching order.

Forecast scoring hides the same complete suffix on each call. We average window scores within sequence and then within each source-label pair. If a source has several labels, its rows remain dependent and should stay together in uncertainty calculations. Residuals are an experimental signal, not established anomaly probabilities.

In [ ]:
from aaai_gait.workflow import anomaly_evaluation
anchor,_=normal_anchor(x,meta,provenance,settings)
source_scores,metrics=anomaly_evaluation(anchor,x,meta,batch_size=settings.batch_size,alpha=.05)
display(pd.Series(metrics))
forecast_scores,forecast_metrics=anomaly_evaluation(forecast_model,x,meta,batch_size=settings.batch_size,alpha=.05,mode='forecast')
from aaai_gait.robustness import anomaly_bootstrap
display(pd.DataFrame([{'method':'masked',**metrics,**anomaly_bootstrap(source_scores,repeats=200)},
                      {'method':'forecast',**forecast_metrics,**anomaly_bootstrap(forecast_scores,repeats=200)}]))
from aaai_gait.workflow import write_json
from aaai_gait.normal_distance import normal_distance
distance_scores,distance_metrics=normal_distance(anchor,x,meta,batch_size=settings.batch_size)
display(pd.Series(distance_metrics))
source_scores.to_csv(provenance['run_dir']/'masked_anomaly_sources.csv',index=False)
forecast_scores.to_csv(provenance['run_dir']/'forecast_anomaly_sources.csv',index=False)
write_json(provenance['run_dir']/'forecast_notebook_metrics.json',forecast_metrics)

## 3. Compare a simple normal-space baseline

The notebook also fits a shrinkage covariance model to one mean embedding per normal training source. Squared Mahalanobis distance measures departure from that reference. No abnormal source trains the reference, and normal validation sources set the same finite-sample threshold. This baseline uses a completed observed sequence; it is not a future-coordinate generator. Source aggregation gives long recordings the same fitting weight as short ones.

## 4. Understand a small calibration set

The threshold uses only normal validation sources. The finite-sample upper rank is `ceil((n + 1) * (1 - alpha))`. If the rank exceeds `n`, the threshold is infinity. At a nominal 5% false-alarm target, at least 19 exchangeable normal calibration observations are needed for a finite threshold. The default real split has only six normal validation sources.

An infinite threshold means that this calibration guarantee is unsupported at the chosen resolution. It is not evidence that the detector recognizes every normal gait. Report ranking metrics, calibration size, and this limitation. An empirical 95th percentile can be reported only as an exploratory operating point with uncertain false-alarm behavior. Do not tune a threshold on test labels.

![Calibration](images/anomaly.svg)

In [ ]:
# Keep the test table hidden during development. Inspect training/validation distributions.
visible=source_scores[source_scores.split.isin(['train','val'])]
fig,ax=plt.subplots(figsize=(8,3))
for label,rows in visible.groupby('label'):ax.scatter([label]*len(rows),rows.score,alpha=.65,label=label)
ax.set_ylabel('Latent KL surprise');ax.tick_params(axis='x',rotation=20);ax.set_title('Synthetic check' if settings.demo else 'Development-source scores');plt.show()

## 5. Localize mismatch without claiming causation

A heatmap describes where the model made larger prediction errors. A bright knee region can arise from unusual motion, an occluded knee, a pose-estimator error, or a camera change. Inspect the walking video and pose confidence alongside the heatmap. It is not a diagnosis or evidence that a joint caused the condition.

In [ ]:
from aaai_gait.model import JOINT_INDICES
chosen=np.flatnonzero(meta.split.eq('val').to_numpy())[0]
residual=forecast_model.latent_residual(torch.as_tensor(x[chosen:chosen+1]),mode='forecast')
from aaai_gait.media import show_window_context
show_window_context(meta.iloc[chosen],manifest)
heat=residual['per_token'][0].numpy()
fig,ax=plt.subplots(figsize=(10,3));im=ax.imshow(np.ma.masked_invalid(heat),aspect='auto',cmap='magma')
ax.set_xticks(range(16),JOINT_INDICES);ax.set_xlabel('BlazePose joint index');ax.set_ylabel('Time patch');ax.set_title('Forecast error on a validation example');fig.colorbar(im,label='Latent KL');plt.show()
plot_motion(x[chosen],title='Synthetic validation example' if settings.demo else 'Validation skeleton');plt.show()

## What to record before a paper run

Save the current annotation hash, source assignments, retained windows, exclusions, software versions, model configuration, seed, and exact command. Keep synthetic artifacts separate. Revisit the video when an apparent anomaly could instead be a crop error, occlusion, or viewpoint change.

The [research review](docs/literature_review.md) explains source methods and adaptations. The [paper plan](docs/paper_plan.md) defines claims, ablations, and limits.